# AdaptiveDepth Phase 5 — post-training evaluation
Evaluates the 500-step student at full depth and across the four trained routes. It compares results with the saved Phase 4 reference, checks full-model retention, and decides whether the project can proceed to Phase 6.

In [ ]:
import base64, os, pathlib, subprocess, sys

REPO_URL = 'https://github.com/VedantPandhare/AdaptiveDepth-Dynamic-Layer-Routing-for-LLMs.git'
base = pathlib.Path('/kaggle/working' if pathlib.Path('/kaggle/working').exists() else '/content')
repo = base / 'AdaptiveDepth-Dynamic-Layer-Routing-for-LLMs'

def read_secret(name):
    try:
        if str(base).startswith('/content'):
            from google.colab import userdata
            return userdata.get(name)
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

if not repo.exists():
    clone_env = os.environ.copy()
    github_token = read_secret('GITHUB_TOKEN')
    if github_token:
        credential = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
        clone_env.update({'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.extraHeader', 'GIT_CONFIG_VALUE_0': f'Authorization: Basic {credential}'})
    clone = subprocess.run(['git', 'clone', REPO_URL, str(repo)], env=clone_env, capture_output=True, text=True)
    if clone.returncode:
        detail = clone.stderr.strip().splitlines()[-1] if clone.stderr.strip() else 'unknown Git error'
        raise RuntimeError(f'Git clone failed: {detail}\nFor a private repository, add a read-only GITHUB_TOKEN to notebook Secrets.')
os.chdir(repo)
print('Repository:', repo)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
subprocess.run(['nvidia-smi'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before evaluating Phase 5'
print('GPU:', torch.cuda.get_device_name(0))

## Provide the trained checkpoint
Choose one source below. For a local ZIP, upload `phase5_student.zip` to `/content` using the Files sidebar before running the next cell. A Google Drive path also works after mounting Drive. For a private Hugging Face model, add `HF_TOKEN` to notebook Secrets.

In [ ]:
import zipfile

LOCAL_CHECKPOINT = ''  # Example: '/content/checkpoint-step-000500'
CHECKPOINT_ZIP = str(base / 'phase5_student.zip')
HF_MODEL_ID = ''  # Optional alternative: 'username/model-name'

if LOCAL_CHECKPOINT and pathlib.Path(LOCAL_CHECKPOINT).exists():
    checkpoint = pathlib.Path(LOCAL_CHECKPOINT).resolve()
elif pathlib.Path(CHECKPOINT_ZIP).exists():
    extract_root = base / 'phase5_student_extracted'
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(CHECKPOINT_ZIP) as archive:
        root = extract_root.resolve()
        for member in archive.infolist():
            target = (root / member.filename).resolve()
            if root != target and root not in target.parents:
                raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
        archive.extractall(root)
    candidates = sorted({path.parent for path in root.rglob('model.safetensors')})
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one checkpoint in ZIP, found {len(candidates)}')
    checkpoint = candidates[0]
elif HF_MODEL_ID:
    from huggingface_hub import snapshot_download
    checkpoint = pathlib.Path(snapshot_download(HF_MODEL_ID, token=read_secret('HF_TOKEN')))
else:
    raise FileNotFoundError('Upload phase5_student.zip, set LOCAL_CHECKPOINT, or set HF_MODEL_ID')

required = ['model.safetensors', 'config.json', 'tokenizer.json']
missing = [name for name in required if not (checkpoint / name).exists()]
if missing:
    raise RuntimeError(f'Incomplete checkpoint; missing: {missing}')
print('Checkpoint:', checkpoint)

In [ ]:
workspace = pathlib.Path('experiments/phase5_evaluation')
command = [
    'adaptivedepth', 'run-phase5-evaluation',
    '--checkpoint', str(checkpoint),
    '--config', 'configs/cloud_t4.yaml',
    '--routes-file', 'configs/phase5_routes.yaml',
    '--dataset', 'data/phase4_eval.jsonl',
    '--reference-file', 'data/phase4_reference_metrics.json',
    '--evaluation-config', 'configs/phase5_evaluation.yaml',
    '--workspace', str(workspace),
    '--resume',
]
subprocess.run(command, check=True)

In [ ]:
print((workspace / 'gate.md').read_text())
archive = pathlib.Path('phase5_evaluation_artifacts.zip')
subprocess.run([sys.executable, '-m', 'zipfile', '-c', str(archive), str(workspace)], check=True)
print('Download:', repo / archive)
print('Upload the ZIP or paste gate.md here for analysis.')